# Online Payment Fraud Detection

This project uses machine learning to identify suspicious online payment transactions. The dataset is downloaded automatically, and several models are compared.

In [ ]:
%pip -q install kagglehub xgboost

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
sns.set_style("whitegrid")

In [ ]:
import kagglehub
folder = kagglehub.dataset_download("ealaxi/paysim1")
files = [f for f in os.listdir(folder) if f.endswith(".csv")]
df = pd.read_csv(os.path.join(folder, files[0]))
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()
df = df.drop(columns=["nameOrig", "nameDest", "isFlaggedFraud"], errors="ignore")
df.describe()

## Fraud distribution

In [ ]:
sns.countplot(data=df, x="isFraud")
plt.xticks([0, 1], ["Normal", "Fraud"])
plt.show()
print("Fraud percentage:", df["isFraud"].mean() * 100)

## Fraud by transaction type

In [ ]:
sns.countplot(data=df, x="type", hue="isFraud")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Transaction amounts

In [ ]:
sns.boxplot(data=df, x="isFraud", y="amount")
plt.yscale("log")
plt.xticks([0, 1], ["Normal", "Fraud"])
plt.show()

## Prepare the data

In [ ]:
X = df.drop("isFraud", axis=1)
y = df["isFraud"]
X = pd.get_dummies(X, columns=["type"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape

## Train models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=150, class_weight="balanced_subsample", n_jobs=-1, random_state=42),
    "Extra Trees": ExtraTreesClassifier(n_estimators=150, class_weight="balanced", n_jobs=-1, random_state=42)
}
trained = {}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    results.append({"Model": name, "Precision": precision_score(y_test, pred, zero_division=0),
                   "Recall": recall_score(y_test, pred, zero_division=0), "F1": f1_score(y_test, pred, zero_division=0),
                   "PR-AUC": average_precision_score(y_test, prob), "ROC-AUC": roc_auc_score(y_test, prob)})
comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False)
comparison

## XGBoost

In [ ]:
from xgboost import XGBClassifier
ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.08, scale_pos_weight=ratio, eval_metric="logloss", random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
trained["XGBoost"] = xgb
pred = xgb.predict(X_test)
prob = xgb.predict_proba(X_test)[:, 1]
results.append({"Model": "XGBoost", "Precision": precision_score(y_test, pred, zero_division=0),
               "Recall": recall_score(y_test, pred, zero_division=0), "F1": f1_score(y_test, pred, zero_division=0),
               "PR-AUC": average_precision_score(y_test, prob), "ROC-AUC": roc_auc_score(y_test, prob)})
comparison = pd.DataFrame(results).sort_values("PR-AUC", ascending=False)
comparison

## Evaluate the best model

In [ ]:
best_name = comparison.iloc[0]["Model"]
best_model = trained[best_name]
pred = best_model.predict(X_test)
prob = best_model.predict_proba(X_test)[:, 1]
print("Best model:", best_name)
print(classification_report(y_test, pred))

In [ ]:
cm = confusion_matrix(y_test, pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Normal", "Fraud"], yticklabels=["Normal", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print("Precision:", precision_score(y_test, pred, zero_division=0))
print("Recall:", recall_score(y_test, pred, zero_division=0))
print("F1:", f1_score(y_test, pred, zero_division=0))
print("PR-AUC:", average_precision_score(y_test, prob))
print("ROC-AUC:", roc_auc_score(y_test, prob))

## Threshold comparison

In [ ]:
rows = []
for t in np.arange(0.1, 1.0, 0.1):
    p = (prob >= t).astype(int)
    rows.append({"Threshold": round(t, 1), "Precision": precision_score(y_test, p, zero_division=0),
                "Recall": recall_score(y_test, p, zero_division=0), "F1": f1_score(y_test, p, zero_division=0)})
threshold_df = pd.DataFrame(rows)
threshold_df

In [ ]:
threshold_df.set_index("Threshold").plot(marker="o", figsize=(8, 5))
plt.show()

## Feature importance

In [ ]:
if hasattr(best_model, "feature_importances_"):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values().tail(20)
    imp.plot(kind="barh", figsize=(8, 6))
    plt.show()

## Save the model

In [ ]:
joblib.dump(best_model, "fraud_detection_model.pkl")
joblib.dump(list(X_train.columns), "fraud_detection_features.pkl")

## Sample prediction

In [ ]:
sample = X_test.iloc[[0]]
p = best_model.predict_proba(sample)[0, 1]
print("Fraud probability:", p)
print("Prediction:", "Fraud" if p >= 0.5 else "Normal")

## Conclusion
The models were trained and compared for detecting fraudulent transactions. Precision, recall, F1-score and PR-AUC help assess their performance on this imbalanced dataset.